# Worksheet: Shapes/Dims, Broadcasting, and `@` (matmul)

Bite-sized questions, sample tensors are
provided so you're not typing setup, just predicting/computing shapes.

For every question: **predict the resulting shape on paper first**, then run
it and check. Shape mistakes are the single most common PyTorch bug -- this
worksheet is about building that "what shape comes out of this" reflex.

No answers included on purpose.

In [ ]:
# The key to DIMENSIONS
#
t = torch.tensor([[1, 2, 3], [4, 5, 6]])


# 1.) get the shape : 
t.shape # torch.Size([2, 3])

# 2.) The dimension you are summing over GOES AWAY in the shape
t.sum()      # 21
t.sum(dim=0) # tensor([5, 7, 9]) : dim = 0 = summing the rows,    dim 0 GOES AWAY in the result
t.sum(dim=1) # tensor([6, 15])   : dim = 1 = summing the columns, dim 1 GOES AWAY in the result

# 3.) dim 1 would go away unless we KEEP IT : (stays a column) 

t.sum(dim=1, keepdim=True) # see below - stays as a COLUMN (dim 1), dim 1 is KEPT
#
# KEEPS the dimension 0
# tensor([[ 6],
#         [15]])v


# SUMMARY : 

t.sum(1).shape                    # torch.Size([2])     dim=0 goes away    
t.sum(dim=0).shape                # torch.Size([3])     dim=1 goes away 
t.sum(dim=0, keepdim=True).shape  # torch.Size([1, 3])  dim=0 becomes 1 --> [1, 3] still 2 dimensions
t.sum(dim=1, keepdim=True).shape  # torch.Size([2, 1])  dim=1 becomes 1 --> [2, 1] still 2 dimensions

tensor([[5, 7, 9]])

# the key to Broadcasting:

```text
line up shapes starting with the trailing dimensions on the right
[2, 3, 3]
   [1, 3]
```

two dimensions are compatible (will broadcast) if

1. the dim is the same

2. one dim is a 1

3. one dim is missing (becomes a 1)

the mechanism : 

1. align from the right

2. pad missing dimensions with a 1

3. stretch dimensions of 1

4. determine output shape - the max of the 2 sizes for that axis/dim

In [2]:
import torch

torch.manual_seed(0)

# a 3-D tensor: 2 planes, 3 rows, 4 columns
t3 = torch.arange(1., 25.).reshape(2, 3, 4)

# a plain matrix
m = torch.arange(1., 13.).reshape(3, 4)

# a vector
v4 = torch.tensor([10., 20., 30., 40.])   # length 4
v3 = torch.tensor([1., 2., 3.])           # length 3

# matmul-shaped operands
a23 = torch.randn(2, 3)
b34 = torch.randn(3, 4)
c24 = torch.randn(2, 4)

# batched matmul operands
batch_a = torch.randn(5, 2, 3)   # 5 matrices, each (2, 3)
batch_b = torch.randn(5, 3, 4)   # 5 matrices, each (3, 4)

# a row vector and a column vector, for outer-product-style broadcasting
row = torch.arange(1., 5.).reshape(1, 4)   # shape (1, 4)
col = torch.arange(1., 4.).reshape(3, 1)   # shape (3, 1)

t3.shape, m.shape, v4.shape, v3.shape

(torch.Size([2, 3, 4]), torch.Size([3, 4]), torch.Size([4]), torch.Size([3]))

## Part 1 -- Dims & Shape

Use `t3` and `m` from the setup cell.

**1.1** — What is `t3.ndim`? Without running it, say what each of the 3 dims represents given how `t3` was built (`reshape(2, 3, 4)`).

In [5]:
# your answer here --> 3
torch.arange(1., 25.).reshape(2, 3, 4).ndim

3

**1.2** — Predict the shape of `t3.reshape(6, 4)` and `t3.reshape(4, 6)`. Then predict: will `t3.reshape(5, 5)` work? Why or why not?

In [ ]:
# your answer here
# will not work because 24 items can't be divided by 5
t3.reshape(5, 5)

**1.3** — Predict the shape of `m.sum(dim=0)` and `m.sum(dim=1)`. State the trick for predicting a reduction's output shape from the input shape.

In [11]:
# your answer here
# m = torch.arange(1., 13.).reshape(3, 4)
torch.arange(1., 13.).reshape(3, 4)

# dim 0  --> [15, 18, 21, 24]   (sum of each column)
# dim 1  --> [10, 26, 42]       (sum of each row)
m.sum(dim=0), m.sum(dim=1)


(tensor([15., 18., 21., 24.]), tensor([10., 26., 42.]))

**1.4** — Predict the shape of `m.mean(dim=1)` vs. `m.mean(dim=1, keepdim=True)`. Why does `keepdim=True` matter if you're about to subtract the mean back from `m`?

In [23]:
# your answer here
# m = torch.arange(1., 13.).reshape(3, 4)
torch.arange(1., 13.).reshape(3, 4)
print(m)

# m.mean(dim=0) --> [5, 6, 7, 8]   (mean of each column)
# m.mean(dim=1) --> [5.5, 11.5, 17.5]       (mean of each row)
m.mean(dim=0), m.mean(dim=1, keepdim=True)

# we need to keep the dimension for broadcasting, so we can subtract the mean from each row of m.

m - m.mean(dim=1, keepdim=True)
# (3, 4)
# (3, 1)  --> does broadcast to (3, 4)
# -------
# (3, 4)
#    (3) does not broadcast

tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])


tensor([[-1.5000, -0.5000,  0.5000,  1.5000],
        [-1.5000, -0.5000,  0.5000,  1.5000],
        [-1.5000, -0.5000,  0.5000,  1.5000]])

**1.5** — Predict the shape of `m - m.mean(dim=1, keepdim=True)`. What would go wrong (or silently do the wrong thing) if you used `m - m.mean(dim=1)` instead (no `keepdim`)?

In [26]:
# your answer here
# see above

(m - m.mean(dim=1, keepdim=True)).shape # 3, 4


torch.Size([3, 4])

**1.6** — Predict `t3.sum(dim=1).shape` and `t3.sum(dim=(0, 2)).shape` (summing over two dims at once).

In [31]:
# your answer here
t3.shape
# t3.sum(dim=1).shape ---> (2, 4)
t3.sum(dim=1).shape
# t3.sum(0, 2).shape ---> (3)  (sum over dim 0 and dim 2)
t3.sum(dim=(0, 2)).shape
# the dimensions you sum over go away


torch.Size([3])

## Part 2 -- Broadcasting

Use `m`, `v4`, `v3`, `row`, `col` from the setup cell.

**2.1** — Predict the shape and result of `m + v4`. State the broadcasting rule that makes this legal (hint: compare `m.shape` and `v4.shape` from the right).

In [ ]:
# your answer here


**2.2** — Will `m + v3` work? Predict whether it errors, and if so, why -- which dimension fails to line up?

In [ ]:
# your answer here


**2.3** — Predict the shape of `row + col`. This is an 'outer' broadcast -- walk through why a `(1, 4)` and a `(3, 1)` combine into something bigger than either input, cell by cell.

In [ ]:
# your answer here


**2.4** — Predict the shape of `a23 + c24` (from the setup cell's matmul operands). Will it work? Why or why not?

In [ ]:
# your answer here
# will not broadcast
# (2, 3)
# (3, 4) dimensions paired from the right are not equal
a23 + c24

**2.5** — You have a batch of images `imgs` with shape `(32, 3, 64, 64)` (batch, channel, height, width) and a per-channel bias `bias` with shape `(3,)`. What shape does `bias` need to be reshaped to so `imgs + bias` broadcasts correctly per-channel? Write the reshape.

In [ ]:
# your answer here
images = torch.randn(32, 3, 64, 64)   # (1, 3, 32, 32)
bias = torch.randn(3)
# (1, 3, 32, 32)
# (1, 3,  1,  1) broadcasts

images + bias.reshape(1, 3, 1, 1)

**2.6** — General rule check: for two shapes to broadcast, you compare their dimensions from the right. State the two conditions that must hold for each aligned pair of dimensions.

In [ ]:
# your answer here
# 1. dimensions must be the same
# 2. one of the dimension must be 1
# 3. or the dimension must be missing - in which case you fill it with 1


## Part 3 -- `@` (matmul)

Use `a23`, `b34`, `c24`, `batch_a`, `batch_b` from the setup cell.

**3.1** — Predict the shape of `a23 @ b34`. State the general rule for what shape `(n, m) @ (m, p)` produces, and what has to be true about the two input shapes for it to be legal at all.

In [55]:
# your answer here
a23 @ b34 # shape will be (2, 4) 

# the general rule is the dimension in the middle must be qeual and it will be removed form the result

# so (n, m) @ (m, p) = (n, p)

tensor([[ 3.1305,  0.9706,  0.3769, -1.7574],
        [ 2.3745,  0.1073,  1.3123, -1.9447]])

**3.2** — Will `a23 @ c24` work? Predict the error (or lack of one) before running it.

In [56]:
# your answer here
# no the shapes are not compatible for matmul - the inner dimensions need to match

**3.3** — What's the difference between `a23 * a23` and `a23 @ a23`? (One of these will actually error for this specific shape -- which, and why?)

In [57]:
# your answer here
# element-wise multiplication verses matmul
# a23 * a23 will work because they are the same shape
# a23 @ a23 will not work because the inner dimensions do not match

a23 * a23

tensor([[2.3747, 0.0861, 4.7471],
        [0.3231, 1.1762, 1.9561]])

**3.4** — Predict the shape of `batch_a @ batch_b`. This is batched matmul -- which dimension is treated as the 'batch' dimension and left alone, and which two dimensions actually get matrix-multiplied?

In [58]:
# your answer here
batch_a.shape, batch_b.shape
(batch_a @ batch_b).shape

torch.Size([5, 2, 4])

**3.5** — Predict the shape of `b34 @ v4`, i.e. a matrix `@` a 1-D vector. PyTorch treats a plain vector specially in matmul -- what shape does the vector effectively get treated as, and what shape comes out?

In [59]:
# your answer here
# 3?
(b34 @ v4).shape

torch.Size([3])

**3.6** — You have `x` with shape `(64, 20)` (a batch of 64 examples, 20 features) and want to run it through a linear layer computing `x @ W + b` that produces 10 outputs per example. What shape must `W` be, and what shape must `b` be for `x @ W + b` to broadcast correctly?

In [ ]:
# your answer here
# W.shape = (20, 10)
# 64, 20 @ 20, 10 = (64, 10)


## Part 4 -- Mixed (put it together)

**4.1** — You compute `scores = x @ W` where `x` is `(64, 20)` and `W` is `(20, 5)`, "
giving `scores` shape `(64, 5)`. You then want to subtract each row's max score "
from itself before softmax (a common numerical-stability trick). Write the "
expression, including any `keepdim` you need, and state the shape at each step.

**4.2** — Given a batch `xb` of shape `(B, 1, 28, 28)` from a `DataLoader`, and a "
`nn.Linear(784, 128)` layer, what has to happen to `xb` before it can be passed "
into the layer, and what shape does it need to become? (Tie this back to `nn.Flatten()`.)

**4.3** — Debugging drill: you get
`RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x784 and 128x10)`.
Which operand is wrong, what shape should it have been, and what's the one-line fix?

In [ ]:
# 4.1 score - score.max(dim=1, keepdim=True) subtracting the max of the row

# 4.2 - (B, 1, 784) --> the 28 x 28 is flattened to a vector of length 784

# 4.3 - (784, 10)
